# 基于凝聚式信息瓶颈（aIB）与序列信息瓶颈（sIB）设计的边缘通信特征提取与压缩实验模型

## 0. 概述

本实验模拟了终端设备采集高维的感知数据，服务器处理数据提取其中的关键信息，并对这些数据进行学习，分别通过凝聚式信息瓶颈（aIB）与序列信息瓶颈（sIB）对数据进行聚类，随后终端设备按照聚类结果对后续数据进行压缩的过程。

## 1. 生成数据与概率分布

假设终端设备采集到的数据X为若干维的均匀分布离散值（可修改为连续值，结果应该类似）的向量，已知X值时Y的后验分布为若干维的二项分布。X到Y的映射方式为通过高斯分布生成若干个最高次数为3的多项式，将X代入，得到的标量$logit_j$映射为$P(Y_j=1)$，用来模拟更复杂的、在终端设备上难以完成的信息提取。X到Y的实际映射方式始终不变，用于后续检验。

首先，随机生成一些X的样本，按照实际映射方式得到一些对应的Y的样本，对采样结果计算X的分布以及已知X时Y的后验分布，用作聚类的样本。

最后，对Y的边缘分布进行统计，如果$P(Y_j=1)$过于接近0或者1会导致Y的信息熵太小，失去预测的意义。

In [8]:
import importlib
import numpy as np

import data as data_mod
import evaluator as eval_mod
import solver as solver_mod

importlib.reload(data_mod)
importlib.reload(eval_mod)
importlib.reload(solver_mod)

from data import (
    generate_data,
    compute_posterior_distribution,
    sample_from_mapping,
    compute_y_marginal_stats,
)
from evaluator import mutual_information, evaluate_transmission
from solver import (
    agglomerative_ib,
    sequential_ib,
    build_cluster_decode_table,
    nearest_centroid_predict,
    knn_predict,
    svm_fit_predict,
)

# ==================== 超参数配置 ====================
N_CLUSTERS = 100       # 聚类后的簇数 K（可按需修改）
N_X = 1000             # 训练样本数
X_DIM = 8            # X 向量维度
Y_DIM = 8            # Y 向量维度
N_TEST = 500          # 测试新样本数
KNN_K = 5             # KNN 近邻数
SVM_C = 10.0          # SVM 惩罚系数 C
SVM_GAMMA = None      # RBF 核参数 gamma（None 表示 1 / X_DIM）
SEED = 1989            # 训练数据随机种子
TEST_SEED = 64       # 测试数据随机种子
# ====================================================

data = generate_data(
    n_x=N_X,
    x_dim=X_DIM,
    x_low=0,
    x_high=5,
    y_dim=Y_DIM,
    max_degree=3,
    seed=SEED,
)

X = data["X"]
P_y_given_x = data["prob_matrix"]

p_xy, p_x, p_y, p_y_given_x_ib = compute_posterior_distribution(P_y_given_x)
i_xy = mutual_information(p_xy)

print(f"原始互信息 I(X; Y) = {i_xy:.4f} bits")

# 统计各维度与总计的 P(Y_i = 1)，确认正负样本分布均衡、后续检验有意义
y_stats = compute_y_marginal_stats(P_y_given_x, threshold=0.5)
print(f"\n=== 训练集边际分布统计 P(Y_i = 1) ===")
print(f"总计平均 P(Y=1) = {y_stats['p_y_total']:.4f} | P(Y|X)>0.5 总占比 = {y_stats['pos_ratio_total']:.2%} | 多数类基线 = {y_stats['majority_baseline_total']:.2%}")
for j in range(Y_DIM):
    print(
        f"  Y[{j}]: P(Y_{j}=1) = {y_stats['p_y_i'][j]:.4f} | "
        f"P>0.5占比 = {y_stats['pos_ratio_i'][j]:.2%} | "
        f"多数类基线 = {y_stats['majority_baseline_i'][j]:.2%}"
    )
imbalanced = np.where((y_stats['pos_ratio_i'] < 0.25) | (y_stats['pos_ratio_i'] > 0.75))[0]
if len(imbalanced) > 0:
    print(f"维度 {imbalanced.tolist()} 类别较不均衡，可调整 SEED 或 logit_scale。")
else:
    print("所有 Y_i 维度分布均衡（P>0.5 占比均在 25%~75% 之间），适合后续检验。")

原始互信息 I(X; Y) = 0.2050 bits

=== 训练集边际分布统计 P(Y_i = 1) ===
总计平均 P(Y=1) = 0.4621 | P(Y|X)>0.5 总占比 = 43.30% | 多数类基线 = 61.20%
  Y[0]: P(Y_0=1) = 0.4533 | P>0.5占比 = 42.60% | 多数类基线 = 57.40%
  Y[1]: P(Y_1=1) = 0.4969 | P>0.5占比 = 50.30% | 多数类基线 = 50.30%
  Y[2]: P(Y_2=1) = 0.4603 | P>0.5占比 = 42.10% | 多数类基线 = 57.90%
  Y[3]: P(Y_3=1) = 0.4009 | P>0.5占比 = 35.30% | 多数类基线 = 64.70%
  Y[4]: P(Y_4=1) = 0.4083 | P>0.5占比 = 35.20% | 多数类基线 = 64.80%
  Y[5]: P(Y_5=1) = 0.6217 | P>0.5占比 = 67.70% | 多数类基线 = 67.70%
  Y[6]: P(Y_6=1) = 0.3834 | P>0.5占比 = 28.20% | 多数类基线 = 71.80%
  Y[7]: P(Y_7=1) = 0.4718 | P>0.5占比 = 45.00% | 多数类基线 = 55.00%
所有 Y_i 维度分布均衡（P>0.5 占比均在 25%~75% 之间），适合后续检验。


## 2. 凝聚式信息瓶颈（aIB）

服务器基于采样的样本，进行特征提取，根据采样数据与特征构建后验概率密度（此处直接使用采样点的固有映射关系替代多次提取，以减少运行时间），而不尝试对实际映射关系进行还原。

随后，服务器通过后验概率密度矩阵对采样的X进行聚类。

In [2]:
aib_result = agglomerative_ib(
    p_x=p_x,
    p_y_given_x=p_y_given_x_ib,
    n_clusters=N_CLUSTERS,
    record_full_hierarchy=True,
)
aib_labels = aib_result["labels"]
aib_i_ty = aib_result["i_ty"]

print(f"aIB 聚类完成: 簇数 = {len(np.unique(aib_labels))}, 样本标签 shape = {aib_labels.shape}")
print(f"aIB 各簇样本数: {np.bincount(aib_labels, minlength=N_CLUSTERS)}")
print(f"aIB 互信息 I(T; Y) = {aib_i_ty:.4f} bits (保留比例 {aib_i_ty / i_xy:.2%})")

aIB 聚类完成: 簇数 = 100, 样本标签 shape = (1000,)
aIB 各簇样本数: [ 6 14 15  9  9 18 12  7 14 19  8 15 16 20 10  8 15 22  4  6 10 14 15 16
 11  8 11  6 11 12 11 16 10  7 15 16 12 18 10  6 11  8  6  6  8  5 12 17
  9 20  7  7 10  9 10  8  7  6  7 10 15 17 13 10  5  8 13  3  6  7  5  3
  8  9  8  5  8  3  5 17 12 13 10 12  7  3 16 13  6  7 12  9 10  4  7  7
  3  6  8  2]
aIB 互信息 I(T; Y) = 0.1586 bits (保留比例 77.38%)


## 3. 序列信息瓶颈（sIB）

In [3]:
sib_result = sequential_ib(
    p_x=p_x,
    p_y_given_x=p_y_given_x_ib,
    n_clusters=N_CLUSTERS,
    max_iter=100,
    n_init=10,
    seed=SEED,
)
sib_labels = sib_result["labels"]
sib_i_ty = sib_result["i_ty"]

# 汇总存放两种算法的聚类结果与后验分布 P(Y|T)，供后续各映射模块使用
clustering_results = {
    "aIB": aib_labels,
    "sIB": sib_labels,
}
cluster_tables = {}
for name, labels in clustering_results.items():
    centroids, y_decode, p_y_given_t = build_cluster_decode_table(
        X, P_y_given_x, labels, n_clusters=N_CLUSTERS
    )
    cluster_tables[name] = {
        "T_centroids": centroids,
        "y_decode_table": y_decode,
        "p_y_given_t": p_y_given_t,
    }

print(f"sIB 聚类完成: 簇数 = {len(np.unique(sib_labels))}, 样本标签 shape = {sib_labels.shape}, 收敛轮数 = {len(sib_result['mi_history']) - 1}")
print(f"sIB 各簇样本数: {np.bincount(sib_labels, minlength=N_CLUSTERS)}")
print(f"sIB 互信息 I(T; Y) = {sib_i_ty:.4f} bits (保留比例 {sib_i_ty / i_xy:.2%})")

sIB 聚类完成: 簇数 = 100, 样本标签 shape = (1000,), 收敛轮数 = 16
sIB 各簇样本数: [11  7 14  8 14  8 13  7  7 12 11  9  9 13 10 12  5  9 10  7  8 19  6 11
  4 11 10  6 10  6  9  8  5  9 14  7 20  9 13  7 13 11 12  4  7 14  3  9
 17  6 15 10 10  7 11  7 11  9  6 13 10  6 10  7  7  6  9 15 12  9  3  7
 15 11 19 16 18  9 17  7 14 16  4  8  9  4  7  3 12  5 18 10 16 12 14 14
 19  6  9  4]
sIB 互信息 I(T; Y) = 0.1616 bits (保留比例 78.84%)


## 4. 最近质心距离映射

在对X进行聚类之后，得到了采样的X与其对应的聚类结果。但X的样本空间的体积比样本数量和聚类数都大了几个数量级，此时还无法确定X如何映射到中间变量。所以，需要确定终端设备在采集到新的数据后如何进行处理。

本实验中，压缩以将数据映射到聚类结果的方式进行。接下来会通过几种不同方式在X的样本空间内进行运算，将新增的X放到它可能属于的聚类中，然后只传输它属于哪个聚类。聚类到Y的映射方式由服务器处理，具体为将聚类内的所有样本的特征概率取平均。

最近质心距离映射是相对而言最简单的方式，服务器计算每个聚类的质心，终端设备下载后只需要计算数据到聚类中心的2-范数，也只需要储存和聚类数量相等的点数。

In [4]:
# 生成新测试样本（沿用同一多项式映射计算真实后验 P(Y|X_test)）
test_data = sample_from_mapping(data, n_samples=N_TEST, seed=TEST_SEED)
X_test = test_data["X"]
P_test = test_data["prob_matrix"]
test_y_stats = compute_y_marginal_stats(P_test, threshold=0.5)
print(
    f"测试集边际统计: 总计平均 P(Y=1) = {test_y_stats['p_y_total']:.4f} | "
    f"P(Y|X)>0.5 总占比 = {test_y_stats['pos_ratio_total']:.2%} | "
    f"多数类基线 = {test_y_stats['majority_baseline_total']:.2%}\n"
)

nearest_centroid_results = {}
for method_name in clustering_results:
    T_centroids = cluster_tables[method_name]["T_centroids"]
    p_y_given_t = cluster_tables[method_name]["p_y_given_t"]

    pred_idx = nearest_centroid_predict(X_test, T_centroids)
    T_test = T_centroids[pred_idx]
    P_pred = p_y_given_t[pred_idx]

    metrics = evaluate_transmission(P_pred, P_test)
    nearest_centroid_results[method_name] = {"T_test": T_test, "P_pred": P_pred, **metrics}

    print(f"=== {method_name} 最近质心映射测试结果 (N_TEST={N_TEST}, K={N_CLUSTERS}) ===")
    print(f"总体还原率: {metrics['overall_acc']:.2%} ({metrics['correct_count']}/{metrics['total_count']})")
    print()

测试集边际统计: 总计平均 P(Y=1) = 0.4536 | P(Y|X)>0.5 总占比 = 41.77% | 多数类基线 = 62.48%

=== aIB 最近质心映射测试结果 (N_TEST=500, K=100) ===
总体还原率: 76.20% (3048/4000)

=== sIB 最近质心映射测试结果 (N_TEST=500, K=100) ===
总体还原率: 76.42% (3057/4000)



## 5. KNN映射

KNN算法是经典的分类算法，但此处性能开销较大。终端设备需要存储聚类信息和所有的样本，并在压缩新数据时找到最近的5个样本点。

In [5]:
knn_results = {}
for method_name, labels in clustering_results.items():
    T_centroids = cluster_tables[method_name]["T_centroids"]
    p_y_given_t = cluster_tables[method_name]["p_y_given_t"]

    train_pred_idx, test_pred_idx = knn_predict(
        X, labels, X_test, n_clusters=N_CLUSTERS, n_neighbors=KNN_K, weighted=True
    )
    train_cls_acc = float((train_pred_idx == labels).mean())
    T_test = T_centroids[test_pred_idx]
    P_pred = p_y_given_t[test_pred_idx]

    metrics = evaluate_transmission(P_pred, P_test)
    knn_results[method_name] = {
        "train_cls_acc": train_cls_acc,
        "T_test": T_test,
        "P_pred": P_pred,
        **metrics,
    }

    print(f"=== {method_name} KNN映射测试结果 (k={KNN_K}, N_TEST={N_TEST}, K={N_CLUSTERS}) ===")
    print(f"训练集簇拟合一致率: {train_cls_acc:.2%}")
    print(f"总体还原率: {metrics['overall_acc']:.2%} ({metrics['correct_count']}/{metrics['total_count']})")
    print()

=== aIB KNN映射测试结果 (k=5, N_TEST=500, K=100) ===
训练集簇拟合一致率: 100.00%
总体还原率: 78.88% (3155/4000)

=== sIB KNN映射测试结果 (k=5, N_TEST=500, K=100) ===
训练集簇拟合一致率: 100.00%
总体还原率: 78.38% (3135/4000)



## 6. SVM映射

SVM的性能和储存开销最大。

In [6]:
svm_results = {}
for method_name, labels in clustering_results.items():
    T_centroids = cluster_tables[method_name]["T_centroids"]
    p_y_given_t = cluster_tables[method_name]["p_y_given_t"]

    train_pred_idx, test_pred_idx = svm_fit_predict(
        X, labels, X_test, n_clusters=N_CLUSTERS, C=SVM_C, gamma=SVM_GAMMA
    )
    train_cls_acc = float((train_pred_idx == labels).mean())
    T_test = T_centroids[test_pred_idx]
    P_pred = p_y_given_t[test_pred_idx]

    metrics = evaluate_transmission(P_pred, P_test)
    svm_results[method_name] = {
        "train_cls_acc": train_cls_acc,
        "T_test": T_test,
        "P_pred": P_pred,
        **metrics,
    }

    print(f"=== {method_name} SVM映射测试结果 (C={SVM_C}, N_TEST={N_TEST}, K={N_CLUSTERS}) ===")
    print(f"训练集簇拟合一致率: {train_cls_acc:.2%}")
    print(f"总体还原率: {metrics['overall_acc']:.2%} ({metrics['correct_count']}/{metrics['total_count']})")
    print()

# ==================== 汇总对比 ====================
print("=== 各映射方式总体还原率汇总 (Y|T 与 Y|X 在 0.5 同侧即正确) ===")
header = f"{'组合':<18} | {'总体还原率':>10} | {'正确数/总数':>12}"
print(header)
print("-" * len(header))
summary_rows = [
    ("aIB + 最近质心", nearest_centroid_results["aIB"]),
    ("sIB + 最近质心", nearest_centroid_results["sIB"]),
    ("aIB + KNN", knn_results["aIB"]),
    ("sIB + KNN", knn_results["sIB"]),
    ("aIB + SVM", svm_results["aIB"]),
    ("sIB + SVM", svm_results["sIB"]),
]
for label_name, m in summary_rows:
    ratio_str = f"{m['correct_count']}/{m['total_count']}"
    print(f"{label_name:<18} | {m['overall_acc']:>9.2%} | {ratio_str:>12}")
print("-" * len(header))
print(f"{'测试集多数类基线':<16} | {test_y_stats['majority_baseline_total']:>9.2%} | {'(各维度盲猜众数)':>12}")

=== aIB SVM映射测试结果 (C=10.0, N_TEST=500, K=100) ===
训练集簇拟合一致率: 100.00%
总体还原率: 79.88% (3195/4000)

=== sIB SVM映射测试结果 (C=10.0, N_TEST=500, K=100) ===
训练集簇拟合一致率: 100.00%
总体还原率: 79.25% (3170/4000)

=== 各映射方式总体还原率汇总 (Y|T 与 Y|X 在 0.5 同侧即正确) ===
组合                 |      总体还原率 |       正确数/总数
----------------------------------------------
aIB + 最近质心         |    76.20% |    3048/4000
sIB + 最近质心         |    76.42% |    3057/4000
aIB + KNN          |    78.88% |    3155/4000
sIB + KNN          |    78.38% |    3135/4000
aIB + SVM          |    79.88% |    3195/4000
sIB + SVM          |    79.25% |    3170/4000
----------------------------------------------
测试集多数类基线         |    62.48% |    (各维度盲猜众数)
